# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhardwaj-ayush03/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*



**Provisional lane: Refresh / Content Opportunity Scoring** (I can confirm or change it until the end of Week 4).

Why this lane: in the starter sample about 54% of pages are labelled declining, and a small group of pages holds most of the exposure (see section 3), so a content team cannot review everything and needs a ranked list of what to look at first. In my Week-2 notebook I built a hand rule (`stale x visible`) and shallow decision trees, and scored them on held-out clients. The trees did not clearly beat the hand rule there (test Precision@50 about 0.52-0.56 vs 0.64, with a baseline rate of 0.542), while the repo's own pipeline report shows a random forest beating a rule (0.74 vs 0.24). So whether a learned ranking really beats a transparent rule is still an open question, and it is worth seven weeks to answer it carefully.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

- **Search question:** Which pages should a content team review first, given limited review capacity?
- **Unit of analysis:** one page (one row per page).
- **Output:** a ranked review queue: a score per page plus a short reason code (for example stale but visible, or declining with demand).
- **Decision it improves:** which pages get reviewed this cycle (for example the top 50).
- **Action:** an SEO / content owner reviews the top pages and decides to refresh, protect, or keep monitoring each one.
- **Cost of a wrong call:** a false positive wastes reviewer time on a page that was fine; a false negative leaves a genuinely declining page unreviewed. Neither outcome shows that a refresh would help.
- **Why data or ML can help at all:** with about 30,000 pages the signal is spread across age, visibility, position, CTR and content depth, which is hard to weigh by hand. It is not just "train a model": the real work is defining a fair label, beating a transparent baseline, and validating on clients the model has not seen.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

The cell below loads the starter CSV and prints the numbers I use to justify this lane.

**What I observed (starter sample, directional only):**

1. About 54.2% of the 30,000 pages (from 32 clients) are labelled declining, so there is far too much to review one by one.
2. Exposure is concentrated: the top 10% of pages hold about 70% of all impressions, so where review time goes matters.
3. My intuition "stale pages are the ones declining" is not supported on its own: stale pages (174 of them) are declining less often (47.1%) than non-stale pages (54.2%). The stale AND visible group is very small (17 pages), although 94% of those are declining. That small n means a simple rule covers few pages and is fragile, which is where a broader, validated score might add value.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((Path(p) for p in [".", "..", "../.."] if (Path(p) / CSV).exists()), None)
if root is None:  # Colab opened straight from GitHub: pull the public starter data
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "_starter"], check=True)
    root = Path("_starter")
df = pd.read_csv(root / CSV)

df["declining"] = df["trend_direction"].str.lower().eq("down")
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
top10 = df["impressions_90d"] >= df["impressions_90d"].quantile(0.9)

print("pages:", len(df), "| clients:", df["client_id"].nunique())
print("declining rate:", round(df["declining"].mean(), 3))
print("declining rate | stale (>=180d since update):", round(df.loc[stale, "declining"].mean(), 3),
      "| not stale:", round(df.loc[~stale, "declining"].mean(), 3), "| stale pages:", int(stale.sum()))
print("stale AND visible pages:", int((stale & visible).sum()),
      "| declining rate:", round(df.loc[stale & visible, "declining"].mean(), 3))
print("share of all impressions held by the top 10% of pages:",
      round(df.loc[top10, "impressions_90d"].sum() / df["impressions_90d"].sum(), 3))

pages: 30000 | clients: 32
declining rate: 0.542
declining rate | stale (>=180d since update): 0.471 | not stale: 0.542 | stale pages: 174
stale AND visible pages: 17 | declining rate: 0.941
share of all impressions held by the top 10% of pages: 0.702


## 4. Careful words: what I can and can't claim

**I can say:** in this anonymized starter sample, I observed that certain signals (age, visibility, position, CTR) are associated with pages labelled declining, and that a ranked list can be compared to a transparent baseline using Precision@K on held-out clients. My results are directional and meant for decision support.

**I cannot say:** that I understand or predict Google's algorithm; that a refresh causes recovery (that would need an experiment); or that these results hold beyond this sample. The label (`trend_direction == "down"`) is a proxy computed from the current window, not a future outcome. My Week-2 results used one split, and Precision@50 on only 50 pages is noisy, so small differences should not be over-read.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.